# First Normal Form (1NF)

In the previous notebook you saw that the book table stores authors and genres
as comma-separated text. That breaks a basic rule of the relational model:
every value should be atomic, a single indivisible piece of data, not a list.
A table that follows this rule is in **First Normal Form (1NF)**. Here you will
bring your Library into 1NF by giving authors and genres their own tables.

## Start from a clean state

Normalization changes the schema, so we start by rebuilding the messy Stage 6
Library. You can run this notebook from the top at any time to start over.

In [ ]:
from rel_db_learning import (
    rebuild_messy_library, get_project_connection, run_sql, run_ddl,
)
from rel_db_learning.dbm_validator import validate_exercise
from rel_db_learning.dbm_validator.exercises import PROJECT_NORM_1NF
from rel_db_learning.widgets.self_assessment import sql_query_check

rebuild_messy_library()
project = get_project_connection()

In [ ]:
run_sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
      AND table_name IN ('author','genre','book_author','book_genre')
""", project)

## The problem once more time

Each book's authors are packed into one text value:

In [ ]:
run_sql("SELECT book_id, title, authors FROM book ORDER BY book_id LIMIT 5", project)

## The fix: give authors and genres their own tables

A book can have many authors, and an author can write many books. That is a
**many-to-many** relationship, and the relational way to model it is a
**junction table**. We will add:

- `author` (author_id, name) and `genre` (genre_id, name)
- `book_author` (book_id, author_id) linking books to their authors
- `book_genre` (book_id, genre_id) linking books to their genres

Each junction's primary key is the pair of foreign keys, so a given book and
author can be linked at most once.

After you build it, your design should look like this. (Open it in pgModeler
to see the relationships.)

<img src="../../pgmodeler/exports/norm_1nf_schema.svg" alt="The 1NF Library schema with author, genre, and the two junction tables" width="800">

## Step 1: create the new tables in pgModeler

1. Open your Stage 6 design in pgModeler and Save As `stage_norm_1nf_design.dbm`.
2. Create table `author` with two columns: `author_id` of type `serial` (this
   makes it an auto-incrementing integer) set as the primary key, and `name` of
   type `varchar`.
3. Create table `genre` the same way: `genre_id` of type `serial` set as the
   primary key, and `name` of type `varchar`.
4. Create junction `book_author` with columns `book_id` and `author_id`, both of
   type `integer`. Make BOTH together the primary key. Add a foreign key
   `book_id -> book.book_id` and a foreign key `author_id -> author.author_id`,
   defined as constraints on these columns (do not use the many-to-many
   relationship tool, to keep the names clean).
5. Create `book_genre` the same way: `book_id` and `genre_id`, composite primary
   key, foreign keys to `book` and `genre`.
6. Do NOT remove `authors` or `genres` from `book` yet. We need them for the migration.
7. Save, then Database -> Diff against your live `my_project`, and paste the
   generated CREATE script below.

The Diff may also list things that are not real changes: it sometimes re-states
the `email_address` and `phone_number` domain CHECK constraints as
`ALTER DOMAIN ... DROP CONSTRAINT` / `ADD CONSTRAINT`, recreates the
`idx_book_title` index, or adds `ALTER TABLE ... OWNER TO` lines. This happens
because pgModeler represents some expressions differently from how PostgreSQL
stores them, so the Diff sees a mismatch where there is none. Running the
`DROP CONSTRAINT` line would fail and roll back the whole step.

Paste only the statements that build the new structure: the four
`CREATE TABLE` statements (`author`, `genre`, `book_author`, `book_genre`) and
their foreign keys (the `ALTER TABLE ... ADD CONSTRAINT ... FOREIGN KEY` lines).
Ignore everything else.

In [ ]:
# Paste the CREATE statements from your pgModeler Diff here:
create_sql = r"""

"""
run_ddl(create_sql, project)

## Step 2: migrate the data

We split each comma-separated value into individual names with PostgreSQL's
`string_to_array` and `unnest`, insert the distinct names into `author` and
`genre`, then link each book to them in the junctions. `trim` removes the spaces
after commas.

In [ ]:
run_ddl(r"""
    INSERT INTO author (name)
    SELECT DISTINCT trim(a)
    FROM book, unnest(string_to_array(authors, ',')) AS a
    WHERE trim(a) <> '';

    INSERT INTO book_author (book_id, author_id)
    SELECT DISTINCT b.book_id, au.author_id
    FROM book b, unnest(string_to_array(b.authors, ',')) AS a
    JOIN author au ON au.name = trim(a)
    WHERE trim(a) <> '';
""", project)

In [ ]:
run_ddl(r"""
    INSERT INTO genre (name)
    SELECT DISTINCT trim(g)
    FROM book, unnest(string_to_array(genres, ',')) AS g
    WHERE trim(g) <> '';

    INSERT INTO book_genre (book_id, genre_id)
    SELECT DISTINCT b.book_id, ge.genre_id
    FROM book b, unnest(string_to_array(b.genres, ',')) AS g
    JOIN genre ge ON ge.name = trim(g)
    WHERE trim(g) <> '';
""", project)

Check the migration. Each book now links to its authors through the junction:

In [ ]:
run_sql(r"""
    SELECT b.title, a.name AS author
    FROM book b
    JOIN book_author ba ON ba.book_id = b.book_id
    JOIN author a ON a.author_id = ba.author_id
    ORDER BY b.title, a.name
    LIMIT 10
""", project)

## Step 3: remove the old columns

The data is safely in the new tables, so remove `authors` and `genres` from `book`.
In pgModeler, delete the `authors` and `genres` columns from `book`, save, Diff
again, and paste the generated DROP COLUMN statements below.

As in Step 1, the Diff may include unrelated domain, index, or owner lines.
Paste only the two `ALTER TABLE book DROP COLUMN` statements and ignore
everything else.

In [ ]:
drop_sql = r"""

"""
run_ddl(drop_sql, project)

## Validate your 1NF design

In [ ]:
validate_exercise(project_db=project, spec=PROJECT_NORM_1NF)

## Querying is now clean

Before, finding books by an author meant fragile substring matching. Now it is a proper join, and you can do things that were a little difficult before like counting books per author:

In [ ]:
run_sql(r"""
    SELECT a.name, COUNT(*) AS books
    FROM author a
    JOIN book_author ba ON ba.author_id = a.author_id
    GROUP BY a.name
    ORDER BY books DESC, a.name
    LIMIT 10
""", project)

## Exercise

Write a query that lists the names of the authors of "The Silmarillion",
ordered by name, using the new tables.

In [ ]:
student_sql = r"""

"""
sql_query_check(
    student_sql=student_sql,
    correct_sql="m05_nb_02_exercise_1",
    connection=project,
)

## What's next

Your Library is now in First Normal Form. The next notebook looks at Second
Normal Form, which concerns tables with composite keys. We use a small
constructed example there, since your Library's single-column keys already
satisfy 2NF.